# FRI-BCI large GPU campaign
The six source runs and 2,430 scored target checkpoints are complete. Run the last cell to read results from Drive. Use the training cell only to resume an incomplete campaign. Target R12 is not used here.


In [ ]:
from pathlib import Path
import subprocess, sys, os, hashlib
REPO = Path('/content/FRI-BCI-GPU-Campaign')
URL = 'https://github.com/Mohnclizzy/FRI-BCI-ML-RL-Decoder-Personal.git'
REF = 'codex/robot-learning-design'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', REF, URL, str(REPO)], check=True)
os.chdir(REPO)
print('Checkout:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
EXPECTED = {'eeg_decoder/gpu_campaign.py': 'dc7b0fd3c1a8ed6f7cbfb06d58902f7f57340064bf4dba6a3873f89ef8679bae', 'configs/gpu_campaign.json': 'c8cd084d02e7fa0f540a4f11df31da28dc1c0b93367d5a290cdd69c850f0e66b', 'configs/gpu_research.json': '1da8f557732c094968cc5b542662377b06c177198077feb880e5123385ddd19d', 'eeg_decoder/gpu_pipeline.py': '8ea200dd7a16d81ee56da12609b64222368f051878bcbf60f70a22e2ceb0aaa1', 'eeg_decoder/neural.py': '6d495b1673317a3a1bdad3c2795ac5d1257e98fd3f7b440a58aa6ff07abb406a', 'eeg_decoder/alignment.py': '043dedee70d0f62da3410938a2a837585678e8fe1134179d290613077bee5c1b', 'eeg_decoder/baseline.py': 'b65f3fe48c083daba6fb631af52a239954debc7e4bcad0c299fbe4a0300eca1d', 'eeg_decoder/risk_gate.py': '8c9ed23def0adf6f9d5226f59482ff4d98af16c1659904b6bb786aff22ecaa68'}
for name, expected in EXPECTED.items():
    assert hashlib.sha256(Path(name).read_text(encoding='utf-8').encode()).hexdigest() == expected, f'{name}: version mismatch; use matching campaign checkout'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
import os
from google.colab import drive
MOUNT = '/content/fri_drive'
if not os.path.ismount(MOUNT):
    drive.mount(MOUNT)
assert os.path.ismount(MOUNT), 'Drive mount did not complete'
WORK = str(Path(MOUNT) / 'MyDrive/FRI_BCI/gpu_campaign_2026_11_12')
print('Saved campaign:', WORK)


In [ ]:
# Run this only to begin or resume training. Completed results stay on Drive.
import subprocess, sys
from pathlib import Path
RUN_TRAINING = False
SHARD = None
if RUN_TRAINING:
    assert 'REPO' in globals() and REPO.is_dir(), 'Run the repository setup cell first'
    assert 'WORK' in globals() and Path(WORK).is_dir(), 'Run the Drive setup cell first'
    command = [sys.executable, '-u', '-m', 'eeg_decoder.gpu_campaign', 'run', '--work-dir', WORK,
               '--cache-dir', '/content/fri_campaign_cache']
    if SHARD is not None:
        command += ['--shard', str(SHARD)]
    log = Path(WORK) / f'launch_{SHARD}.log'
    with log.open('a') as handle:
        with subprocess.Popen(command, cwd=str(REPO), stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end='')
                handle.write(line)
                handle.flush()
            if process.wait():
                raise RuntimeError(f'Campaign failed; inspect {log}')
else:
    print('Training is paused. Set RUN_TRAINING=True only to resume an incomplete campaign.')


In [ ]:
# This cell runs independently after a runtime reset. A GPU and repository checkout are unnecessary.
from pathlib import Path
import json, os
from google.colab import drive
MOUNT = '/content/fri_drive'
if not os.path.ismount(MOUNT):
    drive.mount(MOUNT)
summary_file = Path(MOUNT) / 'MyDrive/FRI_BCI/gpu_campaign_2026_11_12/development_summary.json'
assert summary_file.is_file(), f'Missing results: {summary_file}'
groups = json.loads(summary_file.read_text())['groups']
complete = [(name, row) for name, row in groups.items() if row['complete']]
print(f'Complete recipe groups: {len(complete)}/{len(groups)}')
for mode in ('scratch', 'transfer', 'head_only'):
    name, row = max((item for item in complete if f'/{mode}/' in item[0]),
                    key=lambda item: item[1]['mean_balanced_accuracy'])
    print(f"{mode}: {name}; balanced accuracy {row['mean_balanced_accuracy']:.2%}; "
          f"raw REST active {row['mean_raw_rest_activation']:.2%}; "
          f"imagery missed {row['mean_raw_missed_imagery']:.2%}")
